# Multi-Episode Test (Lightweight)

This notebook mirrors `experiments/run_multi_episode.py` but runs a tiny configuration for testing.
It runs a couple of episodes with a reduced dataset and prints summaries.


In [ ]:
import sys
from pathlib import Path
import torch

ROOT = Path().resolve()
if not (ROOT / 'config').exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

from config import Config, HyperParams
from training.episode import Episode
from experiments.run_utils import build_models, build_optimizers, build_hyperparams

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
Config.DEVICE = device
print('device:', device)


In [ ]:
# Build models/optimizers
models = build_models(device)
optimizers = build_optimizers(models)

# Small hyperparams for test
hyperparams = HyperParams(
    n_samples=200,
    n_paths=20,
    batch_size=64,
    epochs=2,
    simulate_horizon=2,
)
hyperparams.lr = 1e-3
hyperparams.min_lr = 1e-6
hyperparams.warmup_steps = 0
hyperparams.max_steps = 1000


In [ ]:
# Episode settings
n_episodes = 2  # keep small for test
stage_order = [
    ('sdf1', ['sdf_fc1']),
    ('pv', ['policy_value']),
    ('sdf2', ['sdf_fc1']),
    ('fc2', ['fc2']),
]

summaries = []


In [ ]:
for ep in range(n_episodes):
    print(f'=== Episode {ep} ===')
    episode = Episode(
        models=models,
        optimizers=optimizers,
        config=Config,
        hyperparams=hyperparams,
        device=device,
        episode_id=ep,
    )

    data_kwargs = {
        'n_samples': hyperparams.n_samples,
        'n_paths': hyperparams.n_paths if ep == 0 else min(20, hyperparams.n_paths),
        'group_size': 2 if ep == 0 else Config.SIMULATE_GROUP_SIZE,
        'n_branches': Config.BRANCH_NUM,
    }
    simulate_kwargs = {'horizon': hyperparams.simulate_horizon} if ep > 0 else None

    ep_summary = {}
    for stage_name, stage_modules in stage_order:
        summary = episode.run_episode(
            n_epochs=hyperparams.epochs,
            batch_size=hyperparams.batch_size,
            log_interval=10,
            train_modules=stage_modules,
            simulate_kwargs=simulate_kwargs,
            **data_kwargs,
        )
        ep_summary[stage_name] = summary.get('module_summaries', summary)
        print(f'  stage {stage_name} done')

    summaries.append(ep_summary)

print('All episodes done.')
summaries
